# Week 7 — Neural Language Models: Class Work

This notebook covers the Class Demonstrations and Class Activities for Week 7, using sentences from the CBK Annual Report 2024/25 as the working dataset.

In [ ]:
!pip install tensorflow pdfplumber --quiet

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Flatten
import numpy as np

print('TensorFlow version:', tf.__version__)
print('Libraries imported successfully.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 962.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 63.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
TensorFlow version: 2.20.0
Libraries imported successfully.


In [ ]:
from google.colab import files
import pdfplumber

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

text_data = ''
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages[:30]:
        page_text = page.extract_text()
        if page_text:
            text_data += page_text + ' '

print(f'Extracted {len(text_data)} characters from the CBK report.')
print()
print('Sample text:')
print(text_data[:300])

Saving 1084981846_2025 Annual Report.pdf to 1084981846_2025 Annual Report.pdf
Extracted 50083 characters from the CBK report.

Sample text:
To be a World Class Modern Central Bank
“Deepening reforms to enhance macro-economic and
financial sector stability”
A N N U A L R E P O R T
A N D F I N A N C I A L STAT E M E N T S
2024/2025
CENTRAL BANK OF KENYA a
ANNUAL REPORT & FINANCIAL STATEMENTS 2024/25 To be a World Class Modern Central Bank


In [ ]:
import re

raw_sentences = re.split(r'(?<=[.!?])\s+', text_data)
clean_sentences = [s.strip() for s in raw_sentences if 6 <= len(s.split()) <= 14]

texts = clean_sentences[:6]

print('Sample CBK sentences used for Class Demonstrations:')
for i, t in enumerate(texts, 1):
    print(f'{i}. {t}')

Sample CBK sentences used for Class Demonstrations:
1. .......VII
STATEMENT BY THE CHAIRMAN OF THE BOARD OF DIRECTORS............................................................
2. IX
MEMBERS OF THE MONETARY POLICY COMMITTEE................................................................................X
SENIOR MANAGEMENT....................................................................................................................
3. XI
1.0 FINANCIAL STABILITY COMMITTEE ............................................................................................
4. .............19
4.2 The EAC Monetary Cooperation Programme.....................................................................................19
5.0 CENTRAL BANK OPERATIONS...................................................................................................22
5.1 Monetary Operations........................................................................................................
5. The Bank also roll

## Class Demonstration 1 — Building a Simple Neural Network

In [ ]:
model = Sequential([
    Dense(16, activation='relu', input_shape=(4,)),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225 (900.00 B)

 Trainable params: 225 (900.00 B)

 Non-trainable params: 0 (0.00 B)

## Class Demonstration 2 — Understanding the Layers

**What each layer does.**

- **Input Layer** — receives raw data (4 numbers in this case)
- **Hidden Layers (Dense 16, Dense 8)** — find patterns by combining inputs with learned weights
- **Output Layer (Dense 1)** — produces the final prediction

In [ ]:
for i, layer in enumerate(model.layers):
    print(f'Layer {i+1}: {layer.name}')
    print(f'  Output shape: {model.get_layer(index=i).output.shape}')
    print(f'  Parameters:   {layer.count_params()}')
    print()

Layer 1: dense
  Output shape: (None, 16)
  Parameters:   80

Layer 2: dense_1
  Output shape: (None, 8)
  Parameters:   136

Layer 3: dense_2
  Output shape: (None, 1)
  Parameters:   9



## Class Demonstration 3 — Text Tokenization

**Expected Learning:** Convert text into machine-readable format.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

print('Word Index (word -> assigned number):')
print(tokenizer.word_index)
print()
print(f'Total unique words found: {len(tokenizer.word_index)}')

Word Index (word -> assigned number):
{'the': 1, 'of': 2, 'monetary': 3, 'committee': 4, '1': 5, '0': 6, '19': 7, '5': 8, 'bank': 9, 'operations': 10, 'vii': 11, 'statement': 12, 'by': 13, 'chairman': 14, 'board': 15, 'directors': 16, 'ix': 17, 'members': 18, 'policy': 19, 'x': 20, 'senior': 21, 'management': 22, 'xi': 23, 'financial': 24, 'stability': 25, '4': 26, '2': 27, 'eac': 28, 'cooperation': 29, 'programme': 30, 'central': 31, '22': 32, 'also': 33, 'rolled': 34, 'out': 35, 'its': 36, 'strategic': 37, 'plan': 38, 'for': 39, '2024': 40, '2027': 41, 'imports': 42, 'and': 43, 'escalation': 44, 'geopolitical': 45, 'tensions': 46, 'particularly': 47, 'in': 48, 'middle': 49, 'east': 50}

Total unique words found: 50


## Class Demonstration 4 — Converting Text to Sequences

**Expected Learning:** Transform words into numerical representations.

In [ ]:
sequences = tokenizer.texts_to_sequences(texts)

print('Original sentences alongside their number sequences:')
print()
for sentence, seq in zip(texts, sequences):
    print(f'Text:     {sentence}')
    print(f'Sequence: {seq}')
    print()

Original sentences alongside their number sequences:

Text:     .......VII
STATEMENT BY THE CHAIRMAN OF THE BOARD OF DIRECTORS............................................................
Sequence: [11, 12, 13, 1, 14, 2, 1, 15, 2, 16]

Text:     IX
MEMBERS OF THE MONETARY POLICY COMMITTEE................................................................................X
SENIOR MANAGEMENT....................................................................................................................
Sequence: [17, 18, 2, 1, 3, 19, 4, 20, 21, 22]

Text:     XI
1.0 FINANCIAL STABILITY COMMITTEE ............................................................................................
Sequence: [23, 5, 6, 24, 25, 4]

Text:     .............19
4.2 The EAC Monetary Cooperation Programme.....................................................................................19
5.0 CENTRAL BANK OPERATIONS............................................................................................

## Class Demonstration 5 — Simple Prediction Example

 Observe neural network prediction output.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

# Sample training data
# [Income, Credit Score, Existing Loans, Age]
X = np.array([
    [50000, 700, 1, 30],
    [20000, 400, 3, 45],
    [80000, 800, 0, 25],
    [30000, 500, 2, 40]
])

# Labels (1 = Approved, 0 = Rejected)
y = np.array([1, 0, 1, 0])

# Train model
model = LogisticRegression()
model.fit(X, y)

# New customer data
sample = np.array([[45000, 650, 1, 35]])

# Prediction
prediction = model.predict(sample)

print("Customer Data:", sample)
print("Loan Prediction:", prediction)

if prediction == 1:
    print("Result: Loan Approved")
else:
    print("Result: Loan Rejected")

Customer Data: [[45000   650     1    35]]
Loan Prediction: [1]
Result: Loan Approved


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Class Activity 1 — Language Model Comparison

Compare N-Gram Models against Neural Language Models.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Feature': ['Context Understanding', 'Memory Usage', 'Semantic Learning', 'Prediction Accuracy'],
    'N-Gram Model': [
        'Limited to a fixed short window of previous words',
        'Grows very large as vocabulary increases (stores every sequence count)',
        'None — only counts word sequences, no real meaning learned',
        'Poor on unseen word combinations (data sparsity problem)'
    ],
    'Neural Language Model': [
        'Can learn longer and more flexible context through hidden layers',
        'More efficient — learns compact weight matrices instead of storing every count',
        'Learns word meaning through embeddings inside the network',
        'Better generalisation to sentences never seen during training'
    ]
})

comparison

,Feature,N-Gram Model,Neural Language Model
0,Context Understanding,Limited to a fixed short window of previous words,Can learn longer and more flexible context thr...
1,Memory Usage,Grows very large as vocabulary increases (stor...,More efficient — learns compact weight matrice...
2,Semantic Learning,"None — only counts word sequences, no real mea...",Learns word meaning through embeddings inside ...
3,Prediction Accuracy,Poor on unseen word combinations (data sparsit...,Better generalisation to sentences never seen ...


## Class Activity 2 — Neural Network Components

Identify the Input Layer, Hidden Layer, and Output Layer from the model built in Class Demonstration 1.

In [ ]:
print('NEURAL NETWORK COMPONENTS IDENTIFIED')
print('=' * 55)
print()
print('INPUT LAYER')
print('  Shape: (4,) -- receives 4 numbers per example')
print('  Role:  Entry point for raw data into the network')
print()
print('HIDDEN LAYER 1 -- Dense(16, relu)')
print('  Role:  Combines inputs with learned weights to find patterns')
print()
print('HIDDEN LAYER 2 -- Dense(8, relu)')
print('  Role:  Refines patterns found by the first hidden layer')
print()
print('OUTPUT LAYER -- Dense(1, sigmoid)')
print('  Role:  Produces the final prediction as a single value between 0 and 1')

NEURAL NETWORK COMPONENTS IDENTIFIED

INPUT LAYER
  Shape: (4,) -- receives 4 numbers per example
  Role:  Entry point for raw data into the network

HIDDEN LAYER 1 -- Dense(16, relu)
  Role:  Combines inputs with learned weights to find patterns

HIDDEN LAYER 2 -- Dense(8, relu)
  Role:  Refines patterns found by the first hidden layer

OUTPUT LAYER -- Dense(1, sigmoid)
  Role:  Produces the final prediction as a single value between 0 and 1


## Class Activity 3 — NLP Application Discussion

Discuss how Neural Language Models improve real-world NLP applications.

In [ ]:
print('HOW NEURAL LANGUAGE MODELS IMPROVE NLP APPLICATIONS')
print('=' * 60)
print()
print('TRANSLATION SYSTEMS')
print('  Neural models learn meaning, not just word-for-word rules,')
print('  producing much more natural translations (e.g. Google Translate).')
print()
print('CHATBOTS')
print('  Neural models understand context across a conversation rather')
print('  than matching fixed keyword patterns, giving more relevant replies.')
print()
print('SEARCH ENGINES')
print('  Neural models understand the intent behind a search query, not')
print('  just exact keyword matches, returning more relevant results.')
print()
print('VOICE ASSISTANTS')
print('  Neural models predict likely next words and understand spoken')
print('  context, improving speech recognition and natural responses.')

HOW NEURAL LANGUAGE MODELS IMPROVE NLP APPLICATIONS

TRANSLATION SYSTEMS
  Neural models learn meaning, not just word-for-word rules,
  producing much more natural translations (e.g. Google Translate).

CHATBOTS
  Neural models understand context across a conversation rather
  than matching fixed keyword patterns, giving more relevant replies.

SEARCH ENGINES
  Neural models understand the intent behind a search query, not
  just exact keyword matches, returning more relevant results.

VOICE ASSISTANTS
  Neural models predict likely next words and understand spoken
  context, improving speech recognition and natural responses.
